In [91]:
import pandas as pd
import numpy as np
def rename_columns(df):
    return df.rename(columns={
        'Hostname': 'hostname', 'OS': 'os', 'RAM': 'ram_gb', 'cluster ': 'cluster'
    }, errors='raise')

def clean_text(df):
    df = df.copy()
    for col in ['server_id', 'hostname', 'cluster', 'status']:
        df[col] = df[col].str.strip().str.lower()
    for col in ['rack', 'os']:
        df[col] = df[col].str.strip()
    return df

def clean_ram(df):
    df = df.copy()
    df['ram_gb'] = pd.to_numeric(
        df['ram_gb'].astype(str).str.replace('GB', '', case=False, regex=True).str.strip(),
        errors='coerce',
    )
    return df

def clean_dates(df, col='purchase_date'):
    df = df.copy()
    s = df[col].astype(str).str.strip()
    s_norm = s.str.replace(r"[/\-]", ".", regex=True)
    kun_birinchi = pd.to_datetime(s_norm, format="%d.%m.%Y", errors="coerce")
    yil_birinchi = pd.to_datetime(s_norm, format="%Y.%m.%d", errors="coerce")
    oy_nomi      = pd.to_datetime(s,      format="%b %d, %Y", errors="coerce")
    df[col] = kun_birinchi.fillna(yil_birinchi).fillna(oy_nomi)
    return df

def clean_cpu(df):
    df = df.copy()
    differ_ = (df['cpu_cores'] < 1.0) | (df['cpu_cores'] > 100.0)
    df.loc[differ_,'cpu_cores'] = np.nan
    return df

def clean_status(df):
    df = df.copy()
    status_map = pd.Series({
        'active':'active',
        'up':'active',
        'maint':'maintenance',
        'retired':'decommissioned',
        'maintenance':'maintenance',
        'decommissioned':'decommissioned'
    })

    df['status'] = df['status'].map(status_map)
    return df


def clean_server_id(df):
    df = df.copy()
    df['server_id'] = df['server_id'].str.replace(r'^srv(\d{3})$', r'srv-\1', regex=True)
    return df

def clean_dup(df):
    df = df.copy()
    df = df.drop_duplicates(subset='server_id',keep='first')
    return df

# ---- ishga tushirish ----
raw = pd.read_csv('data/raw/servers_inventory.csv')   # bir marta o'qiladi, hech qachon o'zgarmaydi

df = rename_columns(raw)
df = clean_text(df)
df = clean_ram(df)
df = clean_dates(df)
df = clean_cpu(df)
df = clean_status(df)
df = clean_server_id(df)
df = clean_dup(df)
df.to_csv('data/clean/servers_clean.csv',index=False)

In [77]:
df.info()
df.isna().sum()

<class 'pandas.DataFrame'>
Index: 120 entries, 0 to 125
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   server_id      120 non-null    str           
 1   hostname       120 non-null    str           
 2   cluster        120 non-null    str           
 3   rack           118 non-null    str           
 4   os             120 non-null    str           
 5   cpu_cores      116 non-null    float64       
 6   ram_gb         117 non-null    float64       
 7   purchase_date  118 non-null    datetime64[us]
 8   status         120 non-null    str           
dtypes: datetime64[us](1), float64(2), str(6)
memory usage: 9.4 KB


server_id        0
hostname         0
cluster          0
rack             2
os               0
cpu_cores        4
ram_gb           3
purchase_date    2
status           0
dtype: int64

In [9]:
before = df['cpu_cores'].isna().sum()
mask = (df['cpu_cores'] < 1.0) | (df['cpu_cores'] > 100.0)
print(df.loc[mask, ['server_id', 'cpu_cores']])    # nolga aylantirishdan OLDIN ko'r
df.loc[mask, 'cpu_cores'] = np.nan
after = df['cpu_cores'].isna().sum()

print(before, mask.sum(), after)

Empty DataFrame
Columns: [server_id, cpu_cores]
Index: []
5 0 5


In [88]:
df['status'].value_counts(dropna=False)
status_copy = pd.Series({
    'active':'active',
    'up':'active',
    'maint':'maintenance',
    'retired':'decommissioned',
    'maintenance':'maintenance',
    'decommissioned':'decommissioned'
})

a = df.copy()
a['status']  = a['status'].map(status_copy)
a['status'].value_counts(dropna=False)

status
active            111
maintenance        11
decommissioned      4
Name: count, dtype: int64

In [2]:
import json
import pandas as pd

path = 'data/raw/metrics_2026-10-01.jsonl'
good, bad = [], []

with open(path, encoding='utf-8') as f:
    for i, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue
        try:
            good.append(json.loads(line))
        except json.JSONDecodeError:
            bad.append((i, line))

m = pd.DataFrame(good)
print(len(good), len(bad))
m.info()
bad[:3]

11448 7
<class 'pandas.DataFrame'>
RangeIndex: 11448 entries, 0 to 11447
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   server_id     11448 non-null  str   
 1   ts            11448 non-null  object
 2   cpu_pct       11332 non-null  object
 3   mem_pct       11311 non-null  object
 4   disk_pct      11333 non-null  object
 5   net_in_mbps   11320 non-null  object
 6   net_out_mbps  11319 non-null  object
dtypes: object(6), str(1)
memory usage: 626.2+ KB


[(45, '{"server_id": " srv-044 ", "ts": "2026-10-01 00:00:00", "cpu_'),
 (2610,
  '{"server_id": "srv103", "ts": "2026-10-01T05:15:00", "cpu_pct": 44.4, "mem_pct": 51.9, "disk_pct"'),
 (5618,
  '{"server_id": "srv-011", "ts": "01/10/2026 11:45", "cpu_pct": null, "mem_pct": 77.1, "disk_pct": 43.0, "net_in_mbps": 286.2, "ne')]